# README
- Author: RUEI-HSIANG, CHANG（B12303030；AI協助資料整理）
- Created At / Last Modified At: 2026-10-17

## What does this file do?
此檔為使用者要求的精簡閱讀版：保留全部81位候選人及選定的21欄，不補值、不重新分類、不合併不同證據層級的生日或學歷。移除檔案路徑、頁碼、來源網址、查詢與蒐集追蹤欄；完整可稽核版本仍為 candidates_2026_enriched.csv。性別80人有值，其中5人為使用者提供且未獨立查證、11人為媒體補充，葉人文為媒體性別明示描述；不是全部經戶籍資料確認。社群目錄暫存值不納入。空白代表缺值，不是0；首次參選0表示已找到過去參選證據，空白不代表首次=1。年齡是2026減出生年，非選舉當日實歲。此版適合閱讀，正式分析與查證仍應連回完整主表。

## What does this file take?
data/processed/candidates_2026_enriched.csv；SHA-256: 291a31d7588befded91e8dce0caf7a64cc4cecc927bb4c5fbecd218f6c7281b8

## What does this file output?
data/processed/candidates_2026_simple.csv；一人一列、21欄。

## Variable definitions
county_city為選區；candidate_name為姓名；party_at_registration為登記推薦政黨；registration_date為ISO登記日；qualification_status為資格查核標記。birth_date_verified及age_2026_minus_birth_year沿用原主表生日證據（含媒體生日交叉核對的歷史CEC），不是新增身分認證；birth_date_media_supplement、media_birth_year及age_2026_from_media_birth_year為媒體補充。education_profile_as_captured為官方人物頁文字，education_latest_cec與education_latest_cec_year為最近接受的歷史CEC教育類別及年份，education_media_supplement為CNA或在其空白時以遠見補充的媒體學歷文字；遠見補值明標學位待核實，羅佩秦兩項學位並未獨立確認；均非學位認證。incumbency_2026為已蒐集現任資料；first_time_mayor_lifetime與first_time_any_elected_office分別為兩種首次指標。prior_mayor_name_match_years為初步歷史姓名命中年份，不能單獨視為身分確認；media_experience_supplement為媒體經歷文字；candidate_key為與完整資料及證據表連結的唯一鍵。

In [1]:
from pathlib import Path
import hashlib
import pandas as pd


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "AGENTS.md").is_file() and (candidate / "templates").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook within the student project containing AGENTS.md and templates/.")


PROJECT_ROOT = find_project_root()


def project_file(relative_path, area, must_exist=True):
    # Explicit project-relative paths only; never select the first matching file.
    rel = Path(relative_path)
    if rel.is_absolute() or ".." in rel.parts or "<" in str(rel):
        raise ValueError(f"Replace the placeholder with an exact project-relative path: {relative_path}")
    path = (PROJECT_ROOT / rel).resolve()
    allowed = (PROJECT_ROOT / "data" / area).resolve()
    if allowed == PROJECT_ROOT or not allowed.is_relative_to(PROJECT_ROOT):
        raise ValueError(f"data/{area}/ resolves outside the project root.")
    if not path.is_relative_to(allowed):
        raise ValueError(f"Expected a file inside data/{area}/: {relative_path}")
    if must_exist and not path.is_file():
        raise FileNotFoundError(path)
    return path


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def verify_input_sha256(path, expected):
    # Bind this run to the exact file inspected when the notebook was prepared.
    if (not isinstance(expected, str) or len(expected) != 64
            or any(char not in "0123456789abcdefABCDEF" for char in expected)):
        raise ValueError("Fill the expected SHA-256 from the inspected input before loading data.")
    actual = sha256_file(path)
    if actual != expected.lower():
        raise ValueError(f"Input changed: identify the new version before loading {path.name}.")
    return actual


In [2]:
SOURCE='data/processed/candidates_2026_enriched.csv'
EXPECTED_SOURCE_HASH='291a31d7588befded91e8dce0caf7a64cc4cecc927bb4c5fbecd218f6c7281b8'
OUTPUT='data/processed/candidates_2026_simple.csv'
COLUMNS=['candidate_key', 'county_city', 'candidate_name', 'party_at_registration', 'registration_date', 'qualification_status', 'sex_2026', 'birth_date_verified', 'age_2026_minus_birth_year', 'birth_date_media_supplement', 'media_birth_year', 'age_2026_from_media_birth_year', 'education_profile_as_captured', 'education_latest_cec', 'education_latest_cec_year', 'education_media_supplement', 'incumbency_2026', 'first_time_mayor_lifetime', 'first_time_any_elected_office', 'prior_mayor_name_match_years', 'media_experience_supplement']

In [3]:
p=project_file(SOURCE,'processed');verify_input_sha256(p,EXPECTED_SOURCE_HASH)
master=pd.read_csv(p,dtype=str,keep_default_na=False)
assert len(master)==81 and master.candidate_key.is_unique
simple=master.loc[:,COLUMNS].copy()
target=PROJECT_ROOT/OUTPUT
simple.to_csv(target,index=False,encoding='utf-8-sig')
saved=pd.read_csv(target,dtype=str,keep_default_na=False)
pd.testing.assert_frame_equal(saved,master[COLUMNS])
assert saved.candidate_key.is_unique and len(saved)==81
assert not any('source' in c or 'url' in c or c.startswith('file_') or c.endswith('_file') for c in saved.columns)
print('Verified: 81 candidates,',len(saved.columns),'columns; values and row order unchanged.')
print('SHA-256:',sha256_file(target))
display(saved[['county_city','candidate_name','party_at_registration','sex_2026']].head(10))


Verified: 81 candidates, 21 columns; values and row order unchanged.
SHA-256: 17959020575223825ac7be5d4aae238d1cabc3520ae7f89ffa0b657a6d2a2169


,county_city,candidate_name,party_at_registration,sex_2026
0,臺北市,郭璽,台灣麻將最大黨,男
1,臺北市,沈伯洋,民主進步黨,男
2,臺北市,蕭文乾,臺灣SoR無法黨,男
3,臺北市,唐新民,三勢團結促進聯盟,男
4,臺北市,蔣萬安,中國國民黨,男
5,臺北市,林志成,無黨籍及未經政黨推薦,男
6,新北市,蘇巧慧,民主進步黨,女
7,新北市,李四川,中國國民黨,男
8,新北市,蘇輝湟,無黨籍及未經政黨推薦,男
9,桃園市,張善政,中國國民黨,男
